# 07｜标准传感器与相机


> 运行方式：从上到下执行代码单元。控制类 Notebook **没有布尔安全开关**；执行控制单元就会把任务提交给 SDK。
> 需要仿真的课程请先在 FleetDock IDEA 中启动仿真，并确认 `drone1`、`drone2` 等设备在线。


本课需要已启动的 Project AirSim 场景，并要求 `drone1` 配有相机。

![事件、状态与标准传感器](assets/events-sensors.svg)


## 发现传感器

`device.sensors` 是统一 SensorHub。IMU、GNSS、气压计、相机、LiDAR、雷达等都通过标准描述和强类型样本访问。


In [ ]:
from fleetdock_runtime import FleetDock
from fleetdock import SensorKind

FleetDock.init()
drone = FleetDock.devices.drone1
print(drone.connect())

for sensor in drone.sensors.list():
    print(sensor.sensor_id, sensor.kind.value, sensor.source.value)


## 一次读取

`read_once()` 会完成订阅、等待一份样本并自动释放临时订阅。


In [ ]:
imu = drone.sensors.read_once(SensorKind.IMU, timeout_s=10.0)
print("IMU：", imu)

camera = drone.sensors.get(SensorKind.RGB_CAMERA)
frame = drone.sensors.read_once(camera.sensor_id, timeout_s=15.0)
print("相机帧：", frame.sensor_id, frame.width, frame.height, frame.encoding.value)


## 在 Notebook 中显示相机帧

FleetDock 已经把相机数据转换成标准 `ImageFrame`；这里只负责显示，不处理 Project AirSim 协议。


In [ ]:
from IPython.display import display
from PIL import Image

raw_mode = "BGR" if frame.encoding.value == "bgr8" else "RGB"
image = Image.frombytes(
    "RGB",
    (frame.width, frame.height),
    bytes(frame.data),
    "raw",
    raw_mode,
)
display(image)


In [ ]:
print("流状态：", drone.sensors.status(camera.sensor_id))
FleetDock.close()


## 持续订阅

持续处理时使用 `drone.sensors.subscribe(identifier, handler)`；取消函数、速率限制、缓冲和丢帧策略均由 SDK 提供。
